In [19]:
!nvidia-smi

Wed Feb 25 15:31:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [20]:
!pip install ultralytics --quiet

In [21]:
!ls /kaggle/input/datasets/duwipurnamasidik

visdrone-2019-coco-format


In [22]:
!ls /kaggle/input/datasets/duwipurnamasidik/visdrone-2019-coco-format

annotations_VisDrone_train.json  test-challenge  train
annotations_VisDrone_val.json	 test-dev	 val


In [23]:
from pycocotools.coco import COCO
import os
from tqdm import tqdm

dataset_path = "/kaggle/input/datasets/duwipurnamasidik/visdrone-2019-coco-format"
output_path = "/kaggle/working/visdrone_yolo"

os.makedirs(output_path, exist_ok=True)

def convert(split, ann_file):
    coco = COCO(os.path.join(dataset_path, ann_file))
    
    images_dir = os.path.join(dataset_path, split)
    labels_dir = os.path.join(output_path, split, "labels")
    images_out = os.path.join(output_path, split, "images")
    
    os.makedirs(labels_dir, exist_ok=True)
    os.makedirs(images_out, exist_ok=True)

    for img_id in tqdm(coco.getImgIds()):
        img = coco.loadImgs(img_id)[0]
        anns = coco.loadAnns(coco.getAnnIds(imgIds=img_id))
        
        # Copy image path reference (no need to copy actual image in Kaggle)
        label_file = os.path.join(labels_dir, img["file_name"].replace(".jpg", ".txt"))
        
        with open(label_file, "w") as f:
            for ann in anns:
                x, y, w, h = ann["bbox"]
                cx = x + w / 2
                cy = y + h / 2
                cx /= img["width"]
                cy /= img["height"]
                w /= img["width"]
                h /= img["height"]
                
                class_id = ann["category_id"] - 1
                f.write(f"{class_id} {cx} {cy} {w} {h}\n")

convert("train", "annotations_VisDrone_train.json")
convert("val", "annotations_VisDrone_val.json")

print("Conversion complete.")

loading annotations into memory...
Done (t=0.90s)
creating index...
index created!


100%|██████████| 6471/6471 [00:01<00:00, 3501.15it/s]


loading annotations into memory...
Done (t=0.09s)
creating index...
index created!


100%|██████████| 548/548 [00:00<00:00, 2685.49it/s]

Conversion complete.


In [24]:
import yaml

data = {
    "path": "/kaggle/input/datasets/duwipurnamasidik/visdrone-2019-coco-format",
    "train": "train",
    "val": "val",
    "nc": 10,
    "names": [
        "pedestrian",
        "people",
        "bicycle",
        "car",
        "van",
        "truck",
        "tricycle",
        "awning-tricycle",
        "bus",
        "motor"
    ]
}

with open("visdrone_final.yaml", "w") as f:
    yaml.dump(data, f)

print("Final YAML created.")

Final YAML created.


In [26]:
import shutil
import os
from tqdm import tqdm

input_root = "/kaggle/input/datasets/duwipurnamasidik/visdrone-2019-coco-format"
working_root = "/kaggle/working/visdrone_yolo"

for split in ["train", "val"]:
    src = os.path.join(input_root, split)
    dst = os.path.join(working_root, split, "images")
    os.makedirs(dst, exist_ok=True)

    for file in tqdm(os.listdir(src)):
        if file.endswith(".jpg"):
            shutil.copy(os.path.join(src, file), os.path.join(dst, file))

print("Images copied successfully.")

100%|██████████| 548/548 [00:07<00:00, 71.47it/s]

Images copied successfully.


In [27]:
!ls /kaggle/working/visdrone_yolo/train

images	labels


In [28]:
import yaml

data = {
    "path": "/kaggle/working/visdrone_yolo",
    "train": "train/images",
    "val": "val/images",
    "nc": 10,
    "names": [
        "pedestrian",
        "people",
        "bicycle",
        "car",
        "van",
        "truck",
        "tricycle",
        "awning-tricycle",
        "bus",
        "motor"
    ]
}

with open("visdrone_final.yaml", "w") as f:
    yaml.dump(data, f)

print("Final YAML ready.")

Final YAML ready.


In [29]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

model.train(
    data="visdrone_final.yaml",
    epochs=25,
    imgsz=640,
    batch=16,
    workers=2,
    patience=8,
    cos_lr=True,
    cache=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.16 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=visdrone_final.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bd9453e69c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [32]:
import shutil
import glob

# find latest training folder automatically
runs = glob.glob("/kaggle/working/runs/detect/train*")
latest_run = sorted(runs)[-1]

best_model_path = f"{latest_run}/weights/best.pt"

# copy to root output directory
shutil.copy(best_model_path, "/kaggle/working/best_model.pt")

print("Model safely copied to root directory.")

Model safely copied to root directory.
